## Fresnel Rhomb ##

Here we do a first order design of a Fresnel Rhomb, to introduce some of the solid modeling capabilities of prtLab.  A Fresnel Rhomb utilizes total internal reflection (tir) to generate a phase difference between different polarization eigenstates.  

To get started, let's look at the fresnel reflection coefficients for tir:

$$r_s = \frac{\cos(\theta) - i\sqrt{\sin^2(\theta) - n^2}}{\cos(\theta) + i\sqrt{\sin^2(\theta)- n^2}}$$

$$r_p = \frac{n^2 \cos(\theta) - i\sqrt{\sin^2(\theta) - n^2}}{n^2 \cos(\theta) + i\sqrt{\sin^2(\theta) - n^2}}$$


Let's assume the index in the incident medium is 1.5, and in the exit medium it is 1.  So the critical angle is ~41 degrees.  Let's plot the phase difference as a function of theta



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from prtlab import (
    create_fresnel_rhomb_solid,
    jones_retardance,
    n_pk52a_optical_constants,
    plot_prt_system_3d,
    polarization_ray_trace_solid,
    select_dominant_final_ray,
    transform_p_to_jones,
)


In [ ]:
def fresnel_tir_coefficients(theta_deg, n):
    theta = np.deg2rad(theta_deg)
    sqrt_term = np.sqrt(np.sin(theta) ** 2 - n**2)
    cosine = np.cos(theta)
    r_s = (cosine - 1j * sqrt_term) / (cosine + 1j * sqrt_term)
    r_p = (
        (n**2 * cosine - 1j * sqrt_term)
        / (n**2 * cosine + 1j * sqrt_term)
    )
    return r_s, r_p

angles = np.linspace(43, 60, 21)
phase_difference = np.zeros_like(angles)
for index, angle in enumerate(angles):
    r_s, r_p = fresnel_tir_coefficients(angle, 1 / 1.5)
    phase_difference[index] = np.angle(r_s) - np.angle(r_p)

plt.figure()
_ = plt.plot(angles, phase_difference)


A Fresnel Rhomb will have two TIR bounces, so if we can get $\pi$/4  frome each bounce we should be good.  This occurs at roughly 51 degrees.

Now let's turn to the prtLab model of a rhomb.  Example below:  

In [ ]:
wavelength = 0.589  # um
n_glass = 1.5
rhomb = create_fresnel_rhomb_solid(
    n_glass, length=14, height=8, width=8, shear=11.33,
    wavelength=wavelength / 1000, wavelength_units="mm",
)
options = {"maxInteractions": 8, "minAmplitude": 1e-4}
k_in = np.array([0, 0, 1])
x_in = np.array([0, 5, -2])
E_in = np.array([1, 1]) / np.sqrt(2)
ray_output = polarization_ray_trace_solid(
    rhomb, k_in, x_in, E_in, options
)
axis, _ = plot_prt_system_3d(
    rhomb, ray_output,
    {"RaySelection": "dominant", "PostExtend": 4},
)
_ = axis.set_title("Fresnel rhomb polarization evolution")


The rhomb object is specified by the length, height, width, and shear.  The dimensions are scale invariant, what matters are the ratios.  To get to ~51 degrees aoi, we want tan(90-51) = s/l.  If we keep l fixed at 14 (arbitrary), then s should be about 11.33.  You can see that when this is done, I start out with 45 degree polarization, and end up with something that is roughly circular.  

This is not very precise and not using a real glass, so setting up a loop with some actual glass data is worthwhile.  



In [ ]:
shears = np.linspace(10, 12, 101)
coordinate = {
    "type": "doublePole", "a_loc": np.array([0, 0, 1]),
    "x_o": np.array([1, 0, 0]),
}
wavelength = 0.589
n_glass = n_pk52a_optical_constants(wavelength)
retardance_vs_shear = np.zeros_like(shears)
aoi = np.zeros_like(shears)

for index, shear in enumerate(shears):
    rhomb = create_fresnel_rhomb_solid(
        n_glass, length=14, height=8, width=8, shear=shear,
        wavelength=wavelength, wavelength_units="um",
    )
    ray_output = polarization_ray_trace_solid(
        rhomb, k_in, x_in, E_in, options
    )
    final_ray, _ = select_dominant_final_ray(ray_output)
    jones = transform_p_to_jones(
        final_ray.P, k_in, final_ray.k, coordinate
    )
    retardance_vs_shear[index] = jones_retardance(jones)
    normal = ray_output.interactions[1].normal
    aoi[index] = np.rad2deg(
        np.arccos(np.dot([0, 0, 1], normal).real)
    )

plt.figure()
_ = plt.plot(aoi, retardance_vs_shear)


Here the shear has been varied, and the total retardance has been calculated as a function of the angle of incidence on the first tir surface.  It is a little hard to see from the plot, but there are two angles that will give a value very close to pi/2 for this wavelength/index.  I chose a shear of 10.84, which is good enough for this example.  

Another advantage of a rhomb is that it is pretty insensitive to wavelength. We can plot the retardance as a function of wavelength to quantify this:

In [ ]:
shear = 10.84  # eyeball fit
wavelengths = np.linspace(0.4, 0.7, 21)
retardance_vs_wavelength = np.zeros_like(wavelengths)

for index, wavelength in enumerate(wavelengths):
    n_glass = n_pk52a_optical_constants(wavelength)
    rhomb = create_fresnel_rhomb_solid(
        n_glass, length=14, height=8, width=8, shear=shear,
        wavelength=wavelength, wavelength_units="um",
    )
    ray_output = polarization_ray_trace_solid(
        rhomb, k_in, x_in, E_in, options
    )
    final_ray, _ = select_dominant_final_ray(ray_output)
    jones = transform_p_to_jones(
        final_ray.P, k_in, final_ray.k, coordinate
    )
    retardance_vs_wavelength[index] = jones_retardance(jones)

plt.figure()
_ = plt.plot(wavelengths, retardance_vs_wavelength)


The range here is ~.04 radians or 6 milliwaves, which is pretty insenstiive.  

Of course there are other things you can do, such as looking at angle dependence or looking at tolerance errors of the rhomb.  This example is meant to show what can be done with prtLab.  Other solids can be implemented to evaluate things like beamsplitters.  The core machinery of quantifying the interaction at an interface between two materials (the trace interface files) works for any object, but the bookkeeping of tracing to the next surface and defining the surface boundaries is a bit more complicated for 3D objects vs the optical design inspired surface model used for the waveplates.  